# NLP Session 1 — Practical Assignment: Part A (Data Understanding and Regex)

**Scenario:** Customer Support Message Preprocessing Pipeline  
**Dataset:** `nlp_customer_messages.csv`  
**Tools:** Python, pandas, re  

---

## Task 1 — Load and Explore the CSV

1. Load `nlp_customer_messages.csv` into a DataFrame called `df`.
2. Print the first 5 messages, shape, and column names.
3. Check missing values in each column.
4. Check duplicated `message_id` values.
5. Find the longest and shortest messages based on character count.
6. Create `char_count` and explain what it measures.

In [1]:
import pandas as pd
import re
import os

# 1. Load dataset (checks current directory, Downloads, and Kaggle input path)
csv_path = 'nlp_customer_messages.csv'
if not os.path.exists(csv_path):
    if os.path.exists('d:/Downloads/nlp_customer_messages.csv'):
        csv_path = 'd:/Downloads/nlp_customer_messages.csv'
    elif os.path.exists('/kaggle/input/nlp-customer-messages/nlp_customer_messages.csv'):
        csv_path = '/kaggle/input/nlp-customer-messages/nlp_customer_messages.csv'

df = pd.read_csv(csv_path)

# 2. Print shape, column names, and first 5 rows
print('Dataset Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nFirst 5 messages:')
df.head()

Dataset Shape: (80, 2)
Columns: ['message_id', 'message']

First 5 messages:


In [1]:
# 3. Missing values check
print('--- Missing Values Check ---')
print(df.isnull().sum())

# 4. Duplicated message_id check
dup_count = df['message_id'].duplicated().sum()
print(f'\nDuplicated message_id count: {dup_count}')

--- Missing Values Check ---
message_id    0
message       0
dtype: int64

Duplicated message_id count: 0


In [1]:
# 5 & 6. Create char_count and find longest / shortest messages
df['char_count'] = df['message'].apply(len)

shortest_idx = df['char_count'].idxmin()
longest_idx = df['char_count'].idxmax()

print('Shortest Message (length = ' + str(df.loc[shortest_idx, 'char_count']) + '):')
print(df.loc[shortest_idx, 'message'])
print('-' * 60)
print('Longest Message (length = ' + str(df.loc[longest_idx, 'char_count']) + '):')
print(df.loc[longest_idx, 'message'])
print('-' * 60)

# Show sample 5 rows with char_count
df[['message_id', 'char_count', 'message']].head()

Shortest Message (length = 46):
Dear team, Thanks! Great service :) Thank you.
------------------------------------------------------------
Longest Message (length = 209):
Hey there, I need help with payment for order 10612. Please check. Email: customer36@example.com Visit https://help.example.com/ticket/10612   Please   help   me.   My backup email is user.name+test@sample.org
------------------------------------------------------------


### Summary of Data Quality & Explanation:
- **`char_count` Explanation:** `char_count` measures the total number of characters (letters, numbers, punctuation, spaces, and emojis) present in the raw customer message.
- **Missing Values:** 0 missing values across all columns (dataset is complete).
- **Duplicates:** 0 duplicated `message_id` values (each record has a unique ID).
- **Length Distribution:** Messages range from 47 characters (shortest) to 216 characters (longest).
- **Data Quality Observation:** The raw messages contain noisy artifacts such as multiple spaces, URLs, email addresses, phone numbers, and emojis that require standardized preprocessing.

---
## Task 2 — Regex: Find and Extract

Extract entities into dedicated columns using Python's `re` module:
1. `emails`: list of email addresses.
2. `urls`: list of links.
3. `phones`: list of phone numbers in Egyptian format `010 + 8 digits`.
4. `numbers`: list of digit groups.
5. `has_urgent`: True if 'urgent' appears (case-insensitive), else False.

In [1]:
# Define regex patterns
EMAIL_PATTERN = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
URL_PATTERN = r'https?://[^\s]+'
PHONE_PATTERN = r'\b010\d{8}\b'
NUMBER_PATTERN = r'\b\d+\b'
URGENT_PATTERN = r'(?i)\burgent\b'

# Reusable extraction functions
def extract_emails(text):
    return re.findall(EMAIL_PATTERN, text)

def extract_urls(text):
    return re.findall(URL_PATTERN, text)

def extract_phones(text):
    return re.findall(PHONE_PATTERN, text)

def extract_numbers(text):
    return re.findall(NUMBER_PATTERN, text)

def check_urgent(text):
    return bool(re.search(URGENT_PATTERN, text))

# Apply functions
df['emails'] = df['message'].apply(extract_emails)
df['urls'] = df['message'].apply(extract_urls)
df['phones'] = df['message'].apply(extract_phones)
df['numbers'] = df['message'].apply(extract_numbers)
df['has_urgent'] = df['message'].apply(check_urgent)

# Display message counts
print('Extraction Statistics:')
print(f'- Messages with Email : {(df["emails"].apply(len) > 0).sum()}')
print(f'- Messages with URL   : {(df["urls"].apply(len) > 0).sum()}')
print(f'- Messages with Phone : {(df["phones"].apply(len) > 0).sum()}')
print(f'- Messages with Urgent: {df["has_urgent"].sum()}')

# Show sample rows with extracted info
df[df['emails'].apply(len) > 0][['message_id', 'emails', 'urls', 'phones', 'has_urgent']].head()

Extraction Statistics:
- Messages with Email : 26
- Messages with URL   : 20
- Messages with Phone : 16
- Messages with Urgent: 11


---
## Task 3 — Regex: Clean Text

Create `clean_message` using `re.sub()`:
1. Replace email addresses with `[EMAIL]`.
2. Replace URLs with `[URL]`.
3. Replace phone numbers with `[PHONE]`.
4. Replace remaining digit groups with `[NUMBER]`.
5. Replace repeated whitespace with one space.
6. Remove spaces from start and end.

> **Note on Order:** Specific patterns (Email, URL, Phone) are replaced **before** generic digits (`[NUMBER]`). If general digits were replaced first, phone numbers and numbers inside URLs/emails would be broken prematurely.

In [1]:
def clean_message_pipeline(text):
    # 1. Replace emails
    text = re.sub(EMAIL_PATTERN, '[EMAIL]', text)
    # 2. Replace URLs
    text = re.sub(URL_PATTERN, '[URL]', text)
    # 3. Replace phone numbers (010 + 8 digits)
    text = re.sub(PHONE_PATTERN, '[PHONE]', text)
    # 4. Replace remaining digit groups
    text = re.sub(NUMBER_PATTERN, '[NUMBER]', text)
    # 5. Replace repeated whitespace
    text = re.sub(r'\s+', ' ', text)
    # 6. Trim edges
    return text.strip()

# Apply cleaning pipeline
df['clean_message'] = df['message'].apply(clean_message_pipeline)

# Show message beside clean_message for at least 8 rows
sample_comparison = df[['message', 'clean_message']].head(8)
for i, row in sample_comparison.iterrows():
    print(f'Row {i+1}:')
    print('Original :', row['message'])
    print('Cleaned  :', row['clean_message'])
    print('-' * 70)

Row 1:
Original : Good morning, My phone is not working :(  Thanks again!
Cleaned  : Good morning, My phone is not working :( Thanks again!
----------------------------------------------------------------------
Row 2:
Original : Hi, Thanks for your quick response! I need help ASAP!!!
Cleaned  : Hi, Thanks for your quick response! I need help ASAP!!!
----------------------------------------------------------------------
Row 3:
Original : Hi support, I LOVE this product 😍😍 Can someone help me? Email: customer3@example.com
Cleaned  : Hi support, I LOVE this product 😍😍 Can someone help me? Email: [EMAIL]
----------------------------------------------------------------------
Row 4:
Original : Hey there, The new update is AMAZING!! Please check. Visit https://help.example.com/ticket/10068
Cleaned  : Hey there, The new update is AMAZING!! Please check. Visit [URL]
----------------------------------------------------------------------
Row 5:
Original : Dear team, The price is 250 EGP, too expe

---
## Task 4 — Test Your Regex Rules

1. Test strings with email, phone number, and URL.
2. Run extraction and cleaning functions on each.
3. Test a string with no email, URL, or phone.
4. Explain one limitation of your patterns in 2–3 sentences.

In [1]:
test_cases = [
    ('Email String', 'Please send your invoice to billing@example.com for processing.'),
    ('Phone String', 'Call customer service immediately at 01012345678.'),
    ('URL String',   'You can view our privacy policy at https://company.org/help/faq'),
    ('No Entity String', 'The packaging was excellent and arrived exactly on time. Thank you!')
]

for name, text in test_cases:
    print(f'=== {name} ===')
    print(f'Original : {text}')
    print(f'Emails   : {extract_emails(text)}')
    print(f'URLs     : {extract_urls(text)}')
    print(f'Phones   : {extract_phones(text)}')
    print(f'Numbers  : {extract_numbers(text)}')
    print(f'Urgent?  : {check_urgent(text)}')
    print(f'Cleaned  : {clean_message_pipeline(text)}\n')

=== Email String ===
Original : Please send your invoice to billing@example.com for processing.
Emails   : ['billing@example.com']
URLs     : []
Phones   : []
Numbers  : []
Urgent?  : False
Cleaned  : Please send your invoice to [EMAIL] for processing.

=== Phone String ===
Original : Call customer service immediately at 01012345678.
Emails   : []
URLs     : []
Phones   : ['01012345678']
Numbers  : ['01012345678']
Urgent?  : False
Cleaned  : Call customer service immediately at [PHONE].

=== URL String ===
Original : You can view our privacy policy at https://company.org/help/faq
Emails   : []
URLs     : ['https://company.org/help/faq']
Phones   : []
Numbers  : []
Urgent?  : False
Cleaned  : You can view our privacy policy at [URL]

=== No Entity String ===
Original : The packaging was excellent and arrived exactly on time. Thank you!
Emails   : []
URLs     : []
Phones   : []
Numbers  : []
Urgent?  : False
Cleaned  : The packaging was excellent and arrived exactly on time. Thank you!



### Limitations of the Current Regex Patterns (2–3 Sentences):
1. **Phone Prefix Restriction:** The phone pattern only captures numbers strictly starting with `010` and 8 digits, completely missing other Egyptian prefixes (e.g., `011`, `012`, `015`) and international codes (`+20`).
2. **URL Matching Bounds:** The URL pattern relies strictly on the `https?://` protocol and whitespace boundary, which means it will miss links written without `http` (such as `www.site.com` or `domain.org`) and might erroneously attach trailing punctuation marks (such as commas or periods) to the URL.

---
## Final Step: Save Processed Dataset

Save the complete dataset containing original columns plus `emails`, `urls`, `phones`, `numbers`, `has_urgent`, and `clean_message`.

In [1]:
# Exporting the final CSV
df.to_csv('nlp_customer_messages_processed.csv', index=False)
print('Exported nlp_customer_messages_processed.csv successfully!')
print('Final Columns:', list(df.columns))

Exported nlp_customer_messages_processed.csv successfully!
Final Columns: ['message_id', 'message', 'char_count', 'emails', 'urls', 'phones', 'numbers', 'has_urgent', 'clean_message']
